In [34]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content


In [35]:
load_dotenv(override=True)

True

In [36]:
import requests

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [37]:
MODEL_NAME = "gpt-4o-mini"

In [47]:
EMAIL_USER = False

In [39]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

if EMAIL_ADDRESS:
    print ("Email address is set")
else:
    print ("No  email found")

if EMAIL_SMTP_SERVER:
    print ("SMTP Server set")
else:
    print ('SMTP server not found')

if EMAIL_APP_PASSWORD:
    print ('Password is set')
else:
    print ('Password not set')

USE_EMAIL = EMAIL_ADDRESS and EMAIL_SMTP_SERVER and EMAIL_APP_PASSWORD

if USE_EMAIL:
    print ('Email is set up and will try using it')
else:
    print ('Email is not set up; we will send push notification instead')

Email address is set
SMTP Server set
Password is set
Email is set up and will try using it


In [40]:
from email.message import EmailMessage
import smtplib


def send_email (subject, text_body, html_body):
    msg = EmailMessage()
    msg['From'] = EMAIL_ADDRESS
    msg['To'] = EMAIL_ADDRESS
    msg['Subject'] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype='html')

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)

In [41]:
send_email("Testing email send", "This is body. Finger crossed", "<html> <body> <strong> crossed...</strong></body></html>")

In [48]:
def send_message (subject, text_body, html_body):
    if EMAIL_USER:
        send_email(subject, text_body, html_body)
    else:
        print ('Notification')
        push("This is a test message")

In [52]:
send_message("Testing email send", "This is body. Finger crossed", "<html> <body> <strong> crossed...</strong></body></html>")

Notification
Push: This is a test message


In [53]:
intro = """
You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails.
"""
instructions1 = intro + "Your email style is professional, serious with gravitas and credibility."
instructions2 = intro + "Your email style is witty, engaging, and humorous."
instructions3 = intro + "Your email style is concise, to the point, in the style of  a  busy senior  exective."

In [54]:
sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model="gpt-4o-mini")
sales_agent2 = Agent(name="Humorous Sales Agent", instructions=instructions2, model="gpt-4o-mini")
sales_agent3 = Agent(name="Executive Sales Agent", instructions=instructions3, model="gpt-4o-mini")


In [72]:
decision =  """
You pick the best cold sales email from  the given options. 
Imageine you are a customer and pick  the one you most likely to repond to.
Do not give any explanation; reply with the selected email only.
"""

sales_picker = Agent(name="Sales Picker", instructions=decision, model="gpt-4o-mini")

In [66]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Elevate Your Compliance Standards with ComplAI

Dear [Recipient's Name],

I hope this message finds you well.

In today’s rapidly evolving regulatory landscape, ensuring SOC 2 compliance is more critical than ever. As companies face increasing scrutiny, the ability to demonstrate robust security and operational practices is essential not just for regulatory adherence but also for gaining the trust of your clients.

At ComplAI, we understand the complexities associated with SOC 2 compliance. Our AI-powered SaaS tool is designed to simplify these processes, enabling organizations like yours to prepare for audits efficiently while maintaining the highest standards of security and operational integrity.

Key benefits of ComplAI include:

- **Streamlined Compliance Workflows:** Automate and track compliance tasks with ease, ensuring your organization stays on top of requirements.
  
- **Real-time Monitoring:** Gain insights into your compliance posture and identify potential gaps b

In [78]:
import asyncio


message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results =  await asyncio.gather (
        Runner.run(sales_agent1, input=message),
        Runner.run(sales_agent2, input=message),
        Runner.run(sales_agent3, input=message),
    )
outputs = [result.final_output for result in results]

emails = "Cold sles emails from 3 agents:\n\n".join(outputs)

best_email = await Runner.run(sales_picker, input=emails)

print(best_email.final_output)
    

Subject: Is Your SOC 2 Compliance as Chaotic as a Cat in a Room Full of Laser Pointers?

Hi [Recipient's Name],

Let’s face it—navigating SOC 2 compliance can often feel like herding cats. Between all the documentation, audits, and “What even is a control?” conversations, it’s enough to make anyone consider running away to a deserted island (preferably one without Wi-Fi).

But wait! Before you book that one-way ticket, allow me to introduce you to ComplAI, the AI-powered SaaS tool that transforms the wild world of SOC 2 compliance into a well-oiled, efficient machine. Think of us as your trusty compass in the forest of compliance confusion.

With our platform, you can:

- **Automate Documentation:** Wave goodbye to the never-ending paperwork and let our AI do the heavy lifting.
- **Streamline Audits:** Because we all know that preparation is key—like packing an extra pair of sunglasses for that island escape.
- **Stay Compliant:** Sleep easy knowing you’ll be ready for auditors without

In [81]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send an email with the given subject, text body, and HTML body.
    Args:
        subject: The subject of the email
        text_body: The text body of the email
        html_body: The HTML body of the email
    Returns:
        A message indicating that the email was sent successfully
    """
    send_email(subject, text_body, html_body)
    return "Email sent successfully"


In [82]:
send_email_tool.params_json_schema

{'properties': {'subject': {'title': 'Subject', 'type': 'string'},
  'text_body': {'title': 'Text Body', 'type': 'string'},
  'html_body': {'title': 'Html Body', 'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [85]:
from agents import ModelSettings


decision = """
You pick the best cold  sales email from  the given options. 
Imageine you are a customer and pick  the one you most likely to repond to.
Then use your tool to send the email.
"""

require_tool = ModelSettings(tool_choice="required")

sales_sender = Agent(name="Sales Sender", instructions=decision, model="gpt-4o-mini", tools=[send_email_tool], model_settings=require_tool)

In [86]:
import asyncio


message = "Write a cold sales email"

with trace("Sales selection workflow with sending"):
    results =  await asyncio.gather (
        Runner.run(sales_agent1, input=message),
        Runner.run(sales_agent2, input=message),
        Runner.run(sales_agent3, input=message),
    )
outputs = [result.final_output for result in results]

emails = "Cold sles emails from 3 agents:\n\n".join(outputs)

best_email = await Runner.run(sales_sender, input=emails)

print(f"Final response: \n {best_email.final_output}")

Final response: 
 The email titled **"Simplifying Your SOC 2 Compliance Journey with AI"** has been sent successfully. If you need anything else, feel free to ask!


In [84]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [44]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-4o-mini"
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-4o-mini"
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-4o-mini"
)

In [45]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)


Subject: Simplify Your SOC 2 Compliance Process with AI

Hi [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I’m with ComplAI. We specialize in transforming the way organizations handle SOC 2 compliance and prepare for audits.

As you may know, maintaining SOC 2 compliance can be a complex and resource-intensive process. Our AI-powered SaaS tool streamlines this journey, providing your team with the necessary resources and insights to ensure compliance efficiently and effectively.

Key benefits of using ComplAI include:
- **Automated Compliance Tracking:** Stay on top of requirements effortlessly.
- **Customizable Workflows:** Adapt the platform to fit your unique operational processes.
- **Real-time Reporting & Analytics:** Make informed decisions with immediate access to compliance data.

We understand that every organization has its challenges, and I would love the opportunity to discuss how ComplAI can help you simplify your compliance efforts an